### Déploiement du système Intelligent

#### Importation des bibliothèques


In [1]:

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

print("Bibliothèques importées avec succès.")

Bibliothèques importées avec succès.


#### Chargement du modèle et des données afin de récupérer le meilleur modèle entraîné ainsi que le dataset.


In [2]:

modele = joblib.load(
    "C:/Users/LENOVO/Documents/Soutenance/Project_soutenance/models/meilleur_modele.pkl"
)

data = pd.read_excel(
    "C:/Users/LENOVO/Documents/Soutenance/Project_soutenance/data/processed/orbit_dataset_engineering.xlsx"
)

print("Dimensions :", data.shape)

data.head()

Dimensions : (116720, 51)


,id_point,date_collecte,latitude,longitude,fillRate,fillRate_t_minus_1,fillRate_t_minus_2,delta_fillRate_24h,rolling_mean_3d,fillRate_target_t_plus_1,...,quartier_14,rendement_remplissage,pression_citoyenne,ratio_plaintes,indice_meteo,pression_collecte,charge_precollecteur,indice_saturation,indice_priorite,risque_debordement
0,0,2022-01-03,4.001501,9.73807,49.55,33.98,12.98,15.58,32.17,6.00,...,False,133.328571,0,0.0,1714.44,99.10,16.516667,908846.10,0.0,1594.0235
1,0,2022-01-04,4.001501,9.73807,6.00,49.55,33.98,-43.56,29.84,21.43,...,False,15.371429,0,0.0,2052.27,0.00,3.000000,110052.00,0.0,179.0400
2,0,2022-01-05,4.001501,9.73807,21.43,6.00,49.55,15.43,25.66,36.56,...,False,55.434286,0,0.0,1916.20,21.43,10.715000,393069.06,0.0,549.8938
3,0,2022-01-06,4.001501,9.73807,36.56,21.43,6.00,15.13,21.33,52.21,...,False,96.278571,0,0.0,2138.16,73.12,9.140000,670583.52,0.0,779.8248
4,0,2022-01-07,4.001501,9.73807,52.21,36.56,21.43,15.65,36.73,72.72,...,False,139.950000,0,0.0,1740.92,156.63,13.052500,957635.82,0.0,1917.6733


### Préparation des données de prédiction

#### Variables explicatives

In [11]:
X = data[features_modele].copy()

print("Dimensions de X :", X.shape)
print("Variables utilisées :", X.shape[1])

Dimensions de X : (116720, 48)
Variables utilisées : 48


#### Prédiction

In [13]:

predictions = modele.predict(X)

data["fillRate_predit"] = predictions

print("Prédictions réalisées.")

data[
    [
        "id_point",
        "fillRate",
        "fillRate_predit"
    ]
].head()

Prédictions réalisées.


,id_point,fillRate,fillRate_predit
0,0,49.55,9.600721
1,0,6.00,17.379470
2,0,21.43,24.739989
3,0,36.56,50.579721
4,0,52.21,73.755757



#### Création Niveau de priorité

In [15]:

def definir_priorite(x):

    if x < 40:
        return "Faible"

    elif x < 70:
        return "Moyenne"

    elif x < 90:
        return "Élevée"

    else:
        return "Urgente"

data["priorite_prediction"] = (
    data["fillRate_predit"]
    .apply(definir_priorite)
)

data[
    [
        "fillRate_predit",
        "priorite_prediction"
    ]
].head()

,fillRate_predit,priorite_prediction
0,9.600721,Faible
1,17.379470,Faible
2,24.739989,Faible
3,50.579721,Moyenne
4,73.755757,Élevée


#### Génération des recommandations intelligentes

In [16]:

def generer_recommandation(fillrate):

    if fillrate < 40:
        return "Surveillance simple"

    elif fillrate < 70:
        return "Programmer une collecte sous 48 heures"

    elif fillrate < 90:
        return "Planifier une collecte aujourd'hui"

    else:
        return "Collecte immédiate"


data["action_recommandee"] = (
    data["fillRate_predit"]
    .apply(generer_recommandation)
)

print("Recommandations générées.")

Recommandations générées.


#### Affichage

In [17]:

data[
[
"id_point",
"fillRate_predit",
"priorite_prediction",
"action_recommandee"
]
].head(10)

,id_point,fillRate_predit,priorite_prediction,action_recommandee
0,0,9.600721,Faible,Surveillance simple
1,0,17.379470,Faible,Surveillance simple
2,0,24.739989,Faible,Surveillance simple
3,0,50.579721,Moyenne,Programmer une collecte sous 48 heures
4,0,73.755757,Élevée,Planifier une collecte aujourd'hui
5,0,9.987804,Faible,Surveillance simple
6,0,21.483518,Faible,Surveillance simple
7,0,21.292243,Faible,Surveillance simple
8,0,19.026924,Faible,Surveillance simple
9,0,14.010171,Faible,Surveillance simple


#### Préparation des données destinées à Power BI

In [20]:
colonnes_dashboard = [
    "id_point",
    "date_collecte",
    "latitude",
    "longitude",
    "fillRate",
    "fillRate_predit",
    "priorite_prediction",
    "action_recommandee",
    "nb_signalements_citoyens",
    "nb_plaintes",
    "nb_precollecteurs_dispo",
    "jours_depuis_derniere_collecte",
    "capacity_m3",
    "pression_citoyenne",
    "pression_collecte",
    "charge_precollecteur",
    "indice_saturation",
    "indice_priorite",
    "risque_debordement"
]

dashboard = data[colonnes_dashboard].copy()

print("Dimensions du dataset Power BI :", dashboard.shape)
print("\nColonnes :")
print(dashboard.columns.tolist())

dashboard.head()

Dimensions du dataset Power BI : (116720, 19)

Colonnes :
['id_point', 'date_collecte', 'latitude', 'longitude', 'fillRate', 'fillRate_predit', 'priorite_prediction', 'action_recommandee', 'nb_signalements_citoyens', 'nb_plaintes', 'nb_precollecteurs_dispo', 'jours_depuis_derniere_collecte', 'capacity_m3', 'pression_citoyenne', 'pression_collecte', 'charge_precollecteur', 'indice_saturation', 'indice_priorite', 'risque_debordement']


,id_point,date_collecte,latitude,longitude,fillRate,fillRate_predit,priorite_prediction,action_recommandee,nb_signalements_citoyens,nb_plaintes,nb_precollecteurs_dispo,jours_depuis_derniere_collecte,capacity_m3,pression_citoyenne,pression_collecte,charge_precollecteur,indice_saturation,indice_priorite,risque_debordement
0,0,2022-01-03,4.001501,9.73807,49.55,9.600721,Faible,Surveillance simple,0,0,2,2,7,0,99.10,16.516667,908846.10,0.0,1594.0235
1,0,2022-01-04,4.001501,9.73807,6.00,17.379470,Faible,Surveillance simple,0,0,1,0,7,0,0.00,3.000000,110052.00,0.0,179.0400
2,0,2022-01-05,4.001501,9.73807,21.43,24.739989,Faible,Surveillance simple,0,0,1,1,7,0,21.43,10.715000,393069.06,0.0,549.8938
3,0,2022-01-06,4.001501,9.73807,36.56,50.579721,Moyenne,Programmer une collecte sous 48 heures,0,0,3,2,7,0,73.12,9.140000,670583.52,0.0,779.8248
4,0,2022-01-07,4.001501,9.73807,52.21,73.755757,Élevée,Planifier une collecte aujourd'hui,0,0,3,3,7,0,156.63,13.052500,957635.82,0.0,1917.6733


#### Sauvegarde du dataset Power BI

In [21]:

import os

dossier_dashboard = (
    r"C:\Users\LENOVO\Documents\Soutenance"
    r"\Project_soutenance\dashboard"
)

os.makedirs(dossier_dashboard, exist_ok=True)

chemin_dashboard = os.path.join(
    dossier_dashboard,
    "dashboard_dataset.xlsx"
)

dashboard.to_excel(
    chemin_dashboard,
    index=False
)

print("Dataset Power BI enregistré avec succès.")
print("Emplacement :", chemin_dashboard)


Dataset Power BI enregistré avec succès.
Emplacement : C:\Users\LENOVO\Documents\Soutenance\Project_soutenance\dashboard\dashboard_dataset.xlsx


#### Vérification importante

In [22]:
print(
    dashboard[
        [
            "id_point",
            "fillRate",
            "fillRate_predit",
            "priorite_prediction",
            "action_recommandee"
        ]
    ].head(10)
)


   id_point  fillRate  fillRate_predit priorite_prediction  \
0         0     49.55         9.600721              Faible   
1         0      6.00        17.379470              Faible   
2         0     21.43        24.739989              Faible   
3         0     36.56        50.579721             Moyenne   
4         0     52.21        73.755757              Élevée   
5         0     72.72         9.987804              Faible   
6         0     14.71        21.483518              Faible   
7         0     30.54        21.292243              Faible   
8         0     10.78        19.026924              Faible   
9         0      5.05        14.010171              Faible   

                       action_recommandee  
0                     Surveillance simple  
1                     Surveillance simple  
2                     Surveillance simple  
3  Programmer une collecte sous 48 heures  
4      Planifier une collecte aujourd'hui  
5                     Surveillance simple  
6        

In [ ]:

Partie 8 : Préparation des données pour l'application EcoSys

C'est cette partie qui sera utilisée par ton application Web.

L'application n'a pas besoin de toutes les colonnes.

Elle a seulement besoin des informations utiles.
